### Links and Resources
| Function | Description |
|---|---|
| [`ai_query`](https://docs.databricks.com/en/large-language-models/ai-functions.html#ai_query) | Call any served LLM endpoint from SQL |
| [`ai_classify`](https://docs.databricks.com/en/large-language-models/ai-functions.html#ai_classify) | Zero-shot text classification |
| [`ai_extract`](https://docs.databricks.com/en/large-language-models/ai-functions.html#ai_extract) | Pull structured fields from text |
| [`ai_translate`, `ai_summarize`, `ai_analyze_sentiment`, `ai_fix_grammar`, `ai_mask`, `ai_similarity`, `ai_gen`](https://docs.databricks.com/en/large-language-models/ai-functions.html) | Task-specific helpers |
| [Vector Search](https://docs.databricks.com/en/generative-ai/vector-search.html) | `vector_search` SQL function |
| [Foundation Model APIs](https://docs.databricks.com/en/machine-learning/foundation-models/index.html) | Pre-deployed models — DBRX, Llama, Mixtral |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

from pyspark.sql import functions as F

# AI Functions

Databricks ships a family of SQL functions that call LLMs (and a vector index) inline in a query. They turn an unstructured-text problem into something you can join, filter, and aggregate over.

Two families:

1. **Task functions** — `ai_classify`, `ai_extract`, `ai_summarize`, `ai_analyze_sentiment`, `ai_translate`, `ai_fix_grammar`, `ai_mask`, `ai_similarity`. Pre-prompted, you just pass the text.
2. **`ai_query`** — generic call to *any* model serving endpoint. Used when you need a custom prompt, a fine-tuned model, or a private endpoint.

Plus **`vector_search`** for retrieval-augmented patterns.

> **Cost & latency note:** every call goes to a real LLM endpoint. Tokens are billed, latency is hundreds of ms per row. Filter aggressively before applying these — never sprinkle them through a wide query.

In [ ]:
reviews = spark.createDataFrame([
    (1, "The shipping was incredibly fast and the product is amazing!"),
    (2, "Broke after two days. Customer support never replied."),
    (3, "Le produit est arrivé endommagé, je suis très déçu."),
    (4, "Mediocre. Does the job, nothing special."),
    (5, "Outstanding quality, will buy again."),
], ["id", "review"])
reviews.createOrReplaceTempView("reviews")
display(reviews)

## `ai_analyze_sentiment` — single string in, label out

In [ ]:
spark.sql("""
SELECT id, review, ai_analyze_sentiment(review) AS sentiment
FROM reviews
""").show(truncate=False)

## `ai_classify` — zero-shot with your own labels

In [ ]:
spark.sql("""
SELECT id, review,
       ai_classify(review, ARRAY('shipping', 'quality', 'support', 'price')) AS topic
FROM reviews
""").show(truncate=False)

## `ai_translate` — language → language

In [ ]:
spark.sql("""
SELECT id, review,
       ai_translate(review, 'en') AS review_en
FROM reviews
WHERE id IN (3)
""").show(truncate=False)

## `ai_extract` — structured fields from free text

Pass the text and a list of fields you want. Returns a struct.

In [ ]:
spark.sql("""
SELECT id,
       ai_extract(review, ARRAY('issue', 'product_aspect', 'recommendation')) AS extracted
FROM reviews
""").show(truncate=False)

## `ai_summarize` — short summary of a long passage

In [ ]:
long = spark.createDataFrame([
    (1, "I purchased the deluxe edition last month after reading several positive reviews. "
        "Initial impressions were great — the build quality felt premium and setup was a "
        "breeze. However after about three weeks the battery started draining noticeably "
        "faster, and customer support has been unresponsive to my emails. I'd recommend it "
        "with reservations until they sort out the after-sales side."),
], ["id", "text"])
long.createOrReplaceTempView("long")

spark.sql("SELECT id, ai_summarize(text, 25) AS summary FROM long").show(truncate=False)

## `ai_mask` — redact PII

In [ ]:
pii = spark.createDataFrame([
    (1, "Hi, this is Alice Smith calling from +32 471 12 34 56, my email is alice@example.com."),
], ["id", "text"])
pii.createOrReplaceTempView("pii")

spark.sql("""
SELECT ai_mask(text, ARRAY('person', 'email', 'phone')) AS redacted
FROM pii
""").show(truncate=False)

## `ai_similarity` — semantic similarity score (0–1)

In [ ]:
spark.sql("""
SELECT
  ai_similarity('shipping was very fast', 'package arrived quickly')      AS sim_close,
  ai_similarity('shipping was very fast', 'the colour of the box is red') AS sim_far
""").show()

## `ai_query` — custom prompt against any served endpoint

Use this when:
- You want a prompt that the task helpers don't cover.
- You're calling a fine-tuned model on your workspace.
- You need a custom output schema via `responseFormat`.

Signature (simplified):

```sql
ai_query(
  endpoint => 'databricks-dbrx-instruct',
  request  => named_struct(
                'messages',  ARRAY(named_struct('role', 'user', 'content', :prompt)),
                'temperature', 0.0
              ),
  returnType => 'STRING'
)
```

Or the shorthand: `ai_query('endpoint-name', prompt_text)`.

In [ ]:
# Will only work in workspaces with a serving endpoint named below.
# Replace with one available in your workspace, or skip this cell.
ENDPOINT = "databricks-meta-llama-3-1-70b-instruct"

try:
    spark.sql(f"""
    SELECT id,
           ai_query(
             '{ENDPOINT}',
             concat('In one word, what is the dominant emotion in this review? ', review)
           ) AS emotion
    FROM reviews
    """).show(truncate=False)
except Exception as e:
    print("Skipped — endpoint not available:", str(e)[:150])

## `ai_query` with a structured response

Pass `responseFormat` with a JSON schema and Spark parses the response into a struct.

In [ ]:
try:
    spark.sql(f"""
    SELECT id,
           ai_query(
             '{ENDPOINT}',
             concat('Extract sentiment and main_topic as JSON for: ', review),
             responseFormat => 'STRUCT<sentiment: STRING, main_topic: STRING>'
           ) AS parsed
    FROM reviews
    """).show(truncate=False)
except Exception as e:
    print("Skipped — endpoint not available:", str(e)[:150])

## `vector_search` — retrieval over an indexed table

Once you've created a Vector Search index in UC, you can query it with SQL. Returns top-k matches with scores.

```sql
SELECT *
FROM vector_search(
  index => 'main.default.docs_index',
  query => 'how do I reset my password',
  num_results => 5
);
```

The typical pattern is **RAG**: `vector_search` to retrieve passages, `ai_query` to compose an answer using them as context. All inside one SQL query, no glue code.

## Cost-control patterns

- **Filter first.** `WHERE language = 'fr'` before `ai_translate`, etc.
- **Cache results.** AI function output is deterministic-ish per (model, prompt) pair. Materialize into a Delta table; don't re-call.
- **Batch via `MERGE`.** Run AI functions only on rows new since last run.
- **Sample before going wide.** Test on 100 rows, look at the output, *then* run on millions.
- **Watch the endpoint's rate limits.** A query against 10M rows that exceeds the endpoint's RPS will fail or throttle.

## When AI functions are *not* the right tool

- **Hard rules with clear semantics**: a regex or `ai_classify` with 4 labels is not better than 4 `WHEN` clauses if you can write them.
- **Joins / lookups against a known taxonomy**: just join. Don't ask the model.
- **Tight SLAs**: each call is hundreds of ms. Don't put `ai_query` in a sub-second pipeline.

## Takeaways

- Task helpers (`ai_analyze_sentiment`, `ai_classify`, `ai_extract`, `ai_summarize`, `ai_translate`, `ai_mask`, `ai_similarity`) cover the common cases without needing to write a prompt.
- `ai_query` is the escape hatch — any served endpoint, custom prompt, optional structured `responseFormat`.
- `vector_search` + `ai_query` = RAG in pure SQL.
- These are real network calls. Filter, cache, and merge-incrementally — never run them on a full table on every refresh.
- For deterministic, high-volume logic, prefer built-ins or SQL UDFs.